In [1]:
import os
import sys
from pathlib import Path
import json
from typing import Callable

from tqdm import tqdm
import numpy as np
import h5py
import cv2

from scipy import signal
from scipy.signal import decimate

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import braindecode

/home/mike0609king/Documents/projects/hackerthon/schubki2026/venv/lib/python3.11/site-packages/braindecode/models/eegpt.py:497: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = mne.channels.make_standard_montage("standard_1020")
/home/mike0609king/Documents/projects/hackerthon/schubki2026/venv/lib/python3.11/site-packages/braindecode/models/eegpt.py:1452: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = make_standard_montage("standard_1020")


In [5]:
sys.path.append("../src/features")
sys.path.append("../src")

import labram
import biot
import cbramod
import classical_features
import labram_special
import luna
from preprocessing import bandpass, notch, smoothing, ica

import helpers
from helpers import save_features, load_features

In [16]:
BASE_DIR     = Path('../eeg-mind-reading-challenge')
FEAT_DIR     = Path('../data')

STIM_TRN_DIR   = BASE_DIR / Path('stimuli_train')
STIM_TST_DIR   = BASE_DIR / Path('stimuli_test')
TRAIN_DIR      = BASE_DIR / Path('trn')
TEST_DIR       = BASE_DIR / Path('tst')

DEVICE         = "cpu" # cuda or cpu

SFREQ      = 500
NUM_VIDS   = 1102
CH_NAMES   = ["Pz", "P3", "P7", "O1", "Oz", "O2", "P4", "P8", "P1", "P5", "PO7", "PO3", "POz", "PO4", "PO8", "P6", "P2"]

low_filter = 0.1
high_filter = 75.0
notch_freqs= 50.0
do_smoothing = False
smooth_lp = 35.0

# Loading data

In [10]:
def load_subject(sub):
    trn = np.load(f"{TRAIN_DIR}/sub-{sub:02d}_trn.npz")
    stim_ids = np.array(sorted(trn.files, key=int), dtype=int)
    data = np.stack([trn[str(i)] for i in stim_ids])  # (n_trials, 17, 1500)
    return data, stim_ids

def load_subject_test(sub):
    tst_eeg = np.load(f'{TEST_DIR}/sub-{sub:02d}_tst.npy')
    return tst_eeg

def preprocess(x, sfreq):
    x = bandpass(x, sfreq, 0.5, 75.0)
    x = notch(x, sfreq, 50.0)
    return x

X_list, trial_size, stim_list = [], [], []
for sub in tqdm(range(1,6+1)):
    data, stim_ids = load_subject(sub)
    x = preprocess(data, SFREQ)
    X_list.append(x)
    trial_size.append(data.shape[0])
    stim_list.append(stim_ids)
X = np.concatenate(X_list)
stim_ids = np.concatenate(stim_list)

X_test_list, trial_size_test = [], []
for sub in tqdm(range(1,6+1)):
    data = load_subject_test(sub)
    x = preprocess(data, SFREQ)
    trial_size_test.append(data.shape[0])
    X_test_list.append(x)
X_test = np.concatenate(X_test_list)

100%|███████████████████████████████████████████████████████████| 6/6 [00:01<00:00,  4.09it/s]


# EEG Embedding via foundation models

In [17]:
embeddings = [
    (
        FEAT_DIR / Path("schubki_luna"),
        lambda X, ch_names: luna.luna_embed(X, 500, ch_names, batch_size=32, device=DEVICE),
    ),
    (
        FEAT_DIR / Path("schubki_labram"),
        lambda X, ch_names: labram.labram_embed(X, 500, ch_names, batch_size=32, device=DEVICE),
    ),
    (
        FEAT_DIR / Path("schubki_cbramod"),
        lambda X, _: cbramod.cbramod_embed(X, 500, batch_size = 32, device=DEVICE),
    ),
    (
        FEAT_DIR / Path("schubki_labram_special"),
        lambda X, ch_names: labram_special.labram_embed(X, 500, ch_names, batch_size=32, device=DEVICE),
    ),
]

embeddings_test = [
    (
        FEAT_DIR / Path("schubki_luna_test/"),
        lambda X, ch_names: luna.luna_embed(X, 500, ch_names, batch_size=32, device=DEVICE),
    ),
    (
        FEAT_DIR / Path("schubki_labram_test"),
        lambda X, ch_names: labram.labram_embed(X, 500, ch_names, batch_size=32, device=DEVICE),
    ),
    (
        FEAT_DIR / Path("schubki_cbramod_test"),
        lambda X, _: cbramod.cbramod_embed(X, 500, batch_size=32, device=DEVICE),
    ),
    (
        FEAT_DIR / Path("schubki_labram_special_test"),
        lambda X, ch_names: labram_special.labram_embed(X, 500, ch_names, batch_size=32, device=DEVICE),
    ),
]

In [ ]:
for path, embed in tqdm(embeddings):
    print(f"Start {path}")
    path.mkdir(parents=True, exist_ok=True)
    res = embed(X, CH_NAMES)
    prev = 0
    for i, sz in tqdm(enumerate(trial_size), leave=True):
        save_features(
            path / f"sub-{i+1}.h5",
            res[prev:prev+sz],
            stim_ids[prev:prev+sz],
            exists_ok=True
        )
        prev += sz
    save_features(path / "features.h5", res, stim_ids, exists_ok=True)

  0%|                                                                   | 0/4 [00:00<?, ?it/s]

Start ../data/schubki_luna
Kept 17/17 channels.


In [ ]:
# for test data
for path, embed in tqdm(embeddings_test):
    print(f"Start {path}")
    path.mkdir(parents=True, exist_ok=True)
    res = embed(X_test, CH_NAMES)
    prev = 0
    for i, sz in tqdm(enumerate(trial_size_test), leave=True):
        save_features(
            path / f"sub-{i+1}.h5",
            res[prev:prev+sz],
            stim_ids[prev:prev+sz],
            exists_ok=True
        )
        prev += sz
    save_features(path / "features.h5", res, stim_ids, exists_ok=True)